In [1]:
import numpy as np
import pandas as pd

In [2]:
train, valid, test = pd.read_parquet('seznam/charge/train.pqt'), pd.read_parquet('seznam/charge/validation.pqt'), pd.read_parquet('seznam/charge/test.pqt')

In [3]:
test

,client_id,date,sluzba,kc_dobito
400945,100567,2015-07-01,d,25120.00
193066,8653304,2015-07-01,h,5187.28
400924,74900,2015-07-01,d,15153.64
84503,86359,2015-07-01,h,2593.64
252935,9789347,2015-07-01,d,518.10
...,...,...,...,...
21038,9674695,2015-10-01,a,5187.28
256653,23696,2015-10-01,d,77850.02
33980,9773761,2015-10-01,a,1296.82
139829,8672560,2015-10-01,h,2593.64


In [4]:
dobito = pd.read_parquet('seznam/data/dobito.pqt')

In [7]:
dobito.iloc[0]

client_id                8674551
date         2012-08-01 00:00:00
sluzba                         g
kc_dobito                 6280.0
Name: 554345, dtype: object

In [10]:
train.shape, valid.shape, test.shape

((443276, 4), (55410, 4), (55410, 4))

In [6]:
client = pd.read_parquet('seznam/data/client.pqt')

In [7]:
client

,client_id,kraj,obor
0,3901,Vysočina,Vilma
1,3904,Jihomoravský kraj,Leona
2,3907,Zlínský kraj,Vladan
3,3912,Ústecký kraj,Sonja
4,3916,Ústecký kraj,Bohdana
...,...,...,...
73442,9806237,Jihočeský kraj,None
73443,9806258,None,None
73444,9806301,None,None
73445,9806350,Jihomoravský kraj,Leona


In [9]:
# find the distinct values of the column kraj, and count the number of occurences of each value. Including None values
distinct_kraje = client['kraj'].value_counts(dropna=False)
distinct_kraje

kraj
Praha                   19774
Jihomoravský kraj        9493
Středočeský kraj         7582
Moravskoslezský kraj     6405
Jihočeský kraj           3618
Plzeňský kraj            3478
Olomoucký kraj           3361
Královéhradecký kraj     3340
Ústecký kraj             3229
Zlínský kraj             3105
Pardubický kraj          2731
Liberecký kraj           2617
Vysočina                 2038
None                     1414
Karlovarský kraj         1262
Name: count, dtype: int64

In [19]:
# get the maximum frequency of a kraj
max_freq = distinct_kraje.max()
patition_max_freq = max_freq / np.sum(distinct_kraje)

In [20]:
max_freq, patition_max_freq

(19774, 0.2692281509115417)

In [12]:
# split the client to 8:1:1, with random seed 42
random_seed = 42
train_client, valid_client, test_client = np.split(client.sample(frac=1, random_state=random_seed), [int(.8*len(client)), int(.9*len(client))])

/home/ubuntu/miniconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [13]:
train_client

,client_id,kraj,obor
15822,66405,Praha,Vladan
10093,44715,Ústecký kraj,Herbert
41442,1096198,Praha,Dorota
27496,100853,Praha,Vladan
15943,66817,Praha,Milos
...,...,...,...
56006,8724901,Ústecký kraj,Gabriel
35592,930144,Středočeský kraj,Ozzy
71534,9791147,Jihočeský kraj,Zora
14755,62940,Praha,Pink


In [18]:
import os
os.makedirs('seznam/kraj', exist_ok=True)
train_client.to_parquet('seznam/kraj/train.pqt')
valid_client.to_parquet('seznam/kraj/validation.pqt')
test_client.to_parquet('seznam/kraj/test.pqt')

# metadata update
  - name: kraj
    source: kraj/{split}.pqt
    format: parquet
    columns:
      - name: client_id
        dtype: primary_key
      - name: kraj
        dtype: category
      - name: obor
        dtype: category
    evaluation_metric: accuracy
    target_column: kraj
    target_table: Client
    task_type: classification